In [0]:
# High-level description of the notebook workflow:
# 1. Loads raw wiki data from a Delta table and repartitions for parallel processing.
# 2. Defines functions to fetch and enrich QIDs with metadata (label, description, instance QID) from Wikidata API using Pandas UDFs.
# 3. Extracts QIDs from URLs in the wiki data.
# 4. Enriches QIDs with Wikidata metadata and joins this metadata back to the original wiki data.
# 5. Further enriches instance QIDs with their own metadata and joins to the main DataFrame.
# 6. Displays intermediate and final enriched DataFrames.
# 7. Writes the fully enriched wiki data to a Delta table for downstream use.
# 8. Data_visualized in [wiki_data_dashboard]https://dbc-ab30b578-5b4b.cloud.databricks.com/dashboardsv3/01f1170fa26d1cf0a8dad8e7de2608b4/published?o=7474645175957536

In [0]:
# Imports Spark SQL types for defining DataFrame schemas (StructType, StructField, StringType, etc.)
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, BooleanType

# Imports all functions from pyspark.sql.functions for DataFrame transformations (e.g., col, regexp_extract)
from pyspark.sql.functions import * 

# Imports pandas for DataFrame operations in mapInPandas
import pandas as pd

# Imports requests for making HTTP requests to external APIs (e.g., Wikidata)
import requests


In [0]:
# Loads the 'wiki_data_raw' table from the workspace.default schema as a Spark DataFrame and repartitions it into 10 partitions for optimized parallel processing.
wiki_data = spark.table('workspace.default.wiki_data_raw').repartition(10)

_These are Pandas UDFs (a.k.a. vectorized UDFs) in PySpark, introduced to let you write Python functions using pandas and apply them to Spark DataFrames efficiently._

**Helpful Reading**

https://community.databricks.com/t5/technical-blog/understanding-pandas-udf-applyinpandas-and-mapinpandas/ba-p/75717
https://www.databricks.com/blog/2017/10/30/introducing-vectorized-udfs-for-pyspark.html

In [0]:
# This block defines functions and schema for enriching QIDs with Wikidata metadata.
# - batch_fetch(qids): Fetches labels, descriptions, and instance QIDs for a list of QIDs from the Wikidata API.
# - enrich_qids(iterator): Iterator function for Spark mapInPandas, enriches batches of QIDs with metadata using batch_fetch and caching.
# - schema: Specifies the output schema for the enriched DataFrame (qid, label, desc, instance_qid).

def batch_fetch(qids):
    """
    Takes a list of QIDs, returns a dict:
       {qid: (label, instance_qid)}.
    """
    url = (
        "https://www.wikidata.org/w/api.php"
        "?action=wbgetentities"
        f"&ids={'|'.join(qids)}"
        "&languages=en"
        "&format=json"
    )
    headers = {
        "User-Agent": "WikidataBatchFetcher/1.0 (amokwao@gmail.com)"
    }
    resp = requests.get(url, headers=headers, timeout=8)
    if resp.status_code != 200:
        raise RuntimeError(f"HTTP {resp.status_code}")

    out = {}
    for qid, ent in resp.json().get("entities", {}).items():
        label = ent.get("labels", {}).get('en', {}).get("value") 
        desc = ent.get("descriptions", {}).get('en', {}).get("value") 
        p31s  = ent.get("claims", {}).get("P31", [])
        inst_qid = (
            p31s[0]["mainsnak"]["datavalue"]["value"]["id"]  if p31s else None
        )
        
        out[qid] = (label, desc, inst_qid)
    return out


# ***************************** Iterator *****************************************
# - enrich_qids(iterator): Iterator function for Spark mapInPandas, enriches batches of QIDs with metadata using batch_fetch and caching.

def enrich_qids(iterator):
    """
    Receives Pandas batches with column 'qid'.
    Returns batches with 2 new columns: label, instance_qid
    """
    cache = {}         # per-worker in-memory cache  {qid: (label, inst_qid)}

    for pdf in iterator:
        # find QIDs we still need to resolve
        missing = [q for q in pdf["qid"].unique() if q not in cache]
        if missing:
            # resolve in chunks of <= 50 QIDs (API limit)
            for i in range(0, len(missing), 50):
                chunk = missing[i : i + 50]
                try:
                    # fetch labels, descriptions, and instance_qids for missing QIDs and update cache
                    cache.update(batch_fetch(chunk))
                except Exception as e:
                    # fallback: mark failures with the error string
                    cache.update({q: (str(e), None, None) for q in chunk})

        # map cached results back to the batch
        pdf["label"] = pdf["qid"].map(lambda q: cache[q][0])  # assign label from cache
        pdf["desc"] = pdf["qid"].map(lambda q: cache[q][1])   # assign description from cache
        pdf["instance_qid"] = pdf["qid"].map(lambda q: cache[q][2])  # assign instance_qid from cache

        # yield enriched batch with qid, label, desc, and instance_qid columns
        yield pdf[["qid", "label", "desc", "instance_qid"]]


# Defines the schema for the enriched DataFrame with columns: qid, label, desc, and instance_qid
schema = StructType([
    StructField("qid", StringType(), True),
    StructField("label", StringType(), True),
    StructField("desc", StringType(), True),
    StructField("instance_qid", StringType(), True)
])

In [0]:
# Applies enrich_qids to batches of QIDs using Spark mapInPandas, returning a DataFrame with qid, label, desc, and instance_qid columns.
def get_label_value(data):
    return data.mapInPandas(enrich_qids, schema)
def get_label_value(data):
    return data.mapInPandas(enrich_qids, schema) 

In [0]:
# This block extracts QIDs from the 'title_url' column of wiki_data,
# enriches them with Wikidata metadata (label, description, instance_qid),
# and joins the enriched metadata back to the original wiki_data DataFrame.

# Extracts QIDs from the 'title_url' column of wiki_data and creates a DataFrame of non-empty QIDs.
df_qids = (
    wiki_data
    .withColumn("qid", regexp_extract("title_url", r"Q\d+", 0))
    .select("qid")
    .where(col("qid") != "")
)
# Extracts QIDs from the 'title_url' column of wiki_data and creates a DataFrame of unique QIDs for enrichment.
meta_df = get_label_value(df_qids.distinct())

wiki_data_enriched_temp = wiki_data.join(meta_df, wiki_data.title == meta_df.qid, "left")
df_qids = (
    wiki_data
    .withColumn("qid", regexp_extract("title_url", r"Q\d+", 0))
    .select("qid")
    .where(col("qid") != "")
)

# Enriches unique QIDs with Wikidata metadata (label, description, instance_qid) using get_label_value.
meta_df = get_label_value(df_qids.distinct())

# Joins the enriched metadata (meta_df) to the original wiki_data DataFrame on the QID, producing a DataFrame with additional columns: label, desc, and instance_qid.
wiki_data_enriched_temp = wiki_data.join(meta_df, wiki_data.title == meta_df.qid, "left")

In [0]:
# Displays the enriched metadata DataFrame (meta_df) containing QID, label, description, and instance_qid columns.
meta_df.display()


# Displays the enriched wiki_data DataFrame (wiki_data_enriched_temp) with joined metadata columns: label, desc, and instance_qid.
wiki_data_enriched_temp.display()


In [0]:
# Selects the 'instance_qid' column from wiki_data_enriched_temp, renames it to 'qid', filters out null values, and displays the result.
wiki_data_enriched_temp.selectExpr("instance_qid as qid").where('qid is not null').display()

In [0]:
# Selects the 'instance_qid' column from wiki_data_enriched_temp, renames it to 'qid', filters out null values, and assigns the result to df_qids.
df_qids = (
    wiki_data_enriched_temp
    .selectExpr("instance_qid as qid")
    .where('qid is not null')
)


In [0]:
# Enriches unique QIDs with Wikidata metadata, renames columns for instance-level metadata (instance_qid, instance_label, instance_desc)
meta_df = get_label_value(df_qids.distinct()).selectExpr("qid as instance_qid", "label as instance_label", 'desc as instance_desc')

# Joins the enriched instance metadata (meta_df) to the wiki_data_enriched_temp DataFrame on 'instance_qid'
wiki_data_enriched = wiki_data_enriched_temp.join(meta_df, "instance_qid", "left")

In [0]:
# Displays the fully enriched wiki_data DataFrame with joined instance-level metadata columns: instance_label and instance_desc.
wiki_data_enriched.display()

In [0]:
# Writes the fully enriched wiki_data DataFrame to the Delta table 'workspace.default.wiki_data_enriched' in append mode, overwriting the schema if necessary.
wiki_data_enriched.write.format('delta').option("overwriteSchema", "true").mode('append').saveAsTable('workspace.default.wiki_data_enriched')


In [0]:
# Displays the first 5 rows of the 'workspace.default.wiki_data_enriched' Delta table.
spark.table('workspace.default.wiki_data_enriched').limit(5).display()